**[🏠 Course Home](00_START_HERE.ipynb)** | **🤖 [AI-Native](../ai/11_t_test_frequentist_vs_bayesian.ipynb)** | **📖 [Conceptual](../conceptual/11_t_test_frequentist_vs_bayesian.ipynb)** | **🐍 [Python](../python/11_t_test_frequentist_vs_bayesian.ipynb)** | **⬅️ [Prev: Ch 10: Production HMC & Diagnostics](10_mcmc_production_diagnostics.ipynb)** | **➡️ [Next: Ch 12: Bayesian Regression, Counts & GLMs](12_bayesian_regression_and_glms.ipynb)**

*🧭 Part III: Applied Modeling, Regression, Hierarchy & Model Comparison — Chapter 11 of 22*

---

# Chapter 11: Comparing Two Groups — The $t$-Test in Frequentist and Bayesian Paradigms
### *From Null Hypothesis Significance Testing (NHST) to Bayes Factors & Bayesian Estimation (BEST)*

* **Authoritative Track Reference**: [🐍 Python Lab Companion](../python/11_t_test_frequentist_vs_bayesian.ipynb)
* **Conceptual Companion Guide**: [📖 Conceptual Guide: The Two-Sample Comparison](../conceptual/11_t_test_frequentist_vs_bayesian.ipynb)



## 🧭 Executive Summary: Frequentist vs. Bayesian $t$-Test at a Glance

| Feature / Question | Frequentist Paradigm ($t$-test / NHST) | Bayesian Paradigm: Bayes Factor ($BF_{10}$) | Bayesian Paradigm: Estimation (BEST) |
| :--- | :--- | :--- | :--- |
| **Core Question Asked** | *"If the true difference were exactly zero, how improbable would this data be?"* | *"How much more likely is the data under an effect model vs. the null model?"* | *"Given the observed data and prior knowledge, what is the full probability distribution over the effect size?"* |
| **Primary Output** | $t$-statistic, degrees of freedom ($df$), $p$-value, 95% Confidence Interval | Bayes Factor ($BF_{10}$ or $BF_{01}$) | Posterior distribution of $\Delta\mu = \mu_B - \mu_A$, 95% HDI, $P(\mu_B < \mu_A)$, Posterior Cohen's $d$ |
| **Interpretation of Result** | Reject / fail to reject $H_0$. $p < 0.05$ means data is rare under $H_0$. | Continuous ratio of evidence (e.g., $BF_{10} = 8.5$ means data is $8.5\times$ more supportive of $H_1$). | Direct probability statements: *"There is a 97.4% probability Variant B improves latency, with 95% HDI $[-14.2, -1.8]$ ms."* |
| **Evidence FOR the Null?** | ❌ **Impossible**. A high $p$-value ($p > 0.05$) only means *"absence of evidence"*, never *"evidence of absence"*. | ✅ **Yes**. $BF_{01} = 1/BF_{10}$ directly quantifies relative evidence supporting $H_0$. | ✅ **Yes**, via **ROPE** (Region of Practical Equivalence): $P(\|\Delta\mu\| < \epsilon)$. |
| **Interval Meaning** | 95% CI: In 95% of hypothetical repeated identical experiments, the computed interval covers the fixed true $\theta$. | N/A (Hypothesis ratio) | 95% HDI: There is a 95% subjective probability that the true difference lies within this exact interval. |
| **Handling Outliers** | Fragile under heavy tails (Student's $t$ assumes exact normality). | Depends on likelihood specification. | ✅ **Robust**: Can swap Normal likelihood for Student-$t$ likelihood (estimating degrees of freedom $\nu$). |
| **A/B Peeking / Stopping** | ❌ **Invalidates error rates**. Checking sequentially inflates Type I false-positive rate from 5% up to 30%+. | ✅ **Coherent**. Bayes factors update sequentially without penalty. | ✅ **Coherent**. Posterior distribution simply conditions on all observed data to date. |



## Part 1: The Production Scenario & Observed Data

### 1.1 The Engineering Context
Suppose we are backend engineers evaluating a proposed gRPC query optimization.
* **Group A (Control / Baseline)**: Current production microservice.
* **Group B (Treatment / Variant)**: Refactored microservice utilizing batched zero-copy deserialization.

We measure end-to-end request latency (in milliseconds) across independent production requests.
* Latency in real systems is never strictly identical in variance between versions (the optimization might reduce latency variance by eliminating GC pauses).
* Sample sizes might differ slightly ($n_A = 45, n_B = 50$).
* We want to know: **Did the optimization improve latency, and by how much?**

Let's generate synthetic benchmark data with known true parameters to evaluate both methods.



In [ ]:
# Set random seed for exact reproducibility
set.seed(42)

# True parameters (Unknown to the researcher)
true_mu_A <- 145.0      # Baseline mean latency (ms)
true_sigma_A <- 14.0    # Baseline std dev (ms)
n_A <- 40               # Sample size Group A

true_mu_B <- 139.5      # Variant mean latency (ms) -> True diff = -5.5 ms
true_sigma_B <- 18.0    # Variant std dev (ms) -> Heteroscedastic noise!
n_B <- 45               # Sample size Group B

# Generate synthetic observations
y_A <- rnorm(n_A, mean = true_mu_A, sd = true_sigma_A)
y_B <- rnorm(n_B, mean = true_mu_B, sd = true_sigma_B)

cat("==================================================
")
cat("      A/B TESTING OBSERVED DATASET SUMMARY        
")
cat("==================================================
")
cat(sprintf("Group A (Control) : n = %d, Mean = %.2f ms, SD = %.2f ms
", n_A, mean(y_A), sd(y_A)))
cat(sprintf("Group B (Variant) : n = %d, Mean = %.2f ms, SD = %.2f ms
", n_B, mean(y_B), sd(y_B)))
cat(sprintf("Observed Difference (B - A): %.2f ms
", mean(y_B) - mean(y_A)))
cat(sprintf("True Population Difference : %.2f ms
", true_mu_B - true_mu_A))
cat("==================================================
")


### 1.2 Exploratory Data Analysis (EDA)
Before running any statistical test, we visualize the raw distributions. We inspect the center, spread, and individual data points using a combined KDE and jittered strip plot.



In [ ]:
# Exploratory Data Analysis Plots in Base R
par(mfrow = c(1, 2), mar = c(4.5, 4.5, 3.5, 1.5))

# Plot 1: Density Curves
plot(density(y_A), col = "#1f77b4", lwd = 2.5, xlim = c(100, 190),
     main = "Sample Latency Distributions", xlab = "Latency (ms)", ylab = "Density", cex.main = 1.1)
lines(density(y_B), col = "#ff7f0e", lwd = 2.5)
abline(v = mean(y_A), col = "#1f77b4", lty = 2)
abline(v = mean(y_B), col = "#ff7f0e", lty = 2)
legend("topright", legend = c("Group A (Control)", "Group B (Variant)"),
       col = c("#1f77b4", "#ff7f0e"), lwd = 2.5, bty = "n", cex = 0.9)

# Plot 2: Boxplot Comparison
boxplot(list("Group A" = y_A, "Group B" = y_B), col = c("#e6f2ff", "#fff3e0"),
        border = c("#1f77b4", "#ff7f0e"), main = "Latency Distribution by Group",
        ylab = "Latency (ms)", cex.main = 1.1)


---

## Part 2: The Frequentist Paradigm (NHST & Welch's $t$-Test)

### 2.1 The Frequentist Mental Model
In the frequentist framework, parameters are **fixed, immutable constants** of nature. We do not place probability distributions on parameters ($\mu_A, \mu_B$). Instead, probability applies strictly to the **data collection procedure under hypothetical infinite repetition**.

#### The Hypotheses:
* **Null Hypothesis ($H_0$)**: $\mu_A - \mu_B = 0$ (There is zero difference between the groups).
* **Alternative Hypothesis ($H_1$)**: $\mu_A - \mu_B \neq 0$ (Two-sided test).

#### Student's vs. Welch's $t$-Test:
* **Student's $t$-test (1908)**: Assumes both groups share identical variance ($\sigma_A^2 = \sigma_B^2$). It pools variance:
  $$s_{\text{pool}}^2 = \frac{(n_A - 1)s_A^2 + (n_B - 1)s_B^2}{n_A + n_B - 2}$$
  $$t = \frac{\bar{y}_B - \bar{y}_A}{\sqrt{s_{\text{pool}}^2 \left(\frac{1}{n_A} + \frac{1}{n_B}\right)}}$$
* **Welch's $t$-test (1947)**: **Does NOT assume equal variance** (the *Behrens-Fisher problem*). It computes unpooled standard error:
  $$SE_{\Delta} = \sqrt{\frac{s_A^2}{n_A} + \frac{s_B^2}{n_B}}$$
  $$t = \frac{\bar{y}_B - \bar{y}_A}{SE_{\Delta}}$$
  and adjusts the degrees of freedom using the **Welch-Satterthwaite equation**:
  $$df_{\text{Welch}} = \frac{\left(\frac{s_A^2}{n_A} + \frac{s_B^2}{n_B}\right)^2}{\frac{(s_A^2 / n_A)^2}{n_A - 1} + \frac{(s_B^2 / n_B)^2}{n_B - 1}}$$

> [!IMPORTANT]
> **Best Practice Rule:** Always default to **Welch's $t$-test** (`equal_var=False` in Base R stats). When variances are equal, Welch's test loses virtually zero power. When variances differ (which is almost always true in production), Student's test has inflated Type I error rates.



In [ ]:
# Execute Welch's t-test using Base R
res_welch <- t.test(y_B, y_A, var.equal = FALSE)

cat("==================================================
")
cat("      WELCH'S TWO-SAMPLE t-TEST (FREQUENTIST)     
")
cat("==================================================
")
cat(sprintf("t-statistic                       : %.4f
", res_welch$statistic))
cat(sprintf("Degrees of Freedom (Welch-Satt)   : %.2f
", res_welch$parameter))
cat(sprintf("p-value (two-tailed)              : %.4e
", res_welch$p.value))
cat(sprintf("95%% Confidence Interval (B - A)  : [%.3f, %.3f] ms
", 
            res_welch$conf.int[1], res_welch$conf.int[2]))
cat(sprintf("Significant at alpha = 0.05?      : %s
", res_welch$p.value < 0.05))
cat("==================================================
")


### 2.2 Critical Conceptual Breakdown: The 3 Fatal Fallacies of NHST

Even when computed flawlessly, frequentist outputs are notoriously prone to misinterpretation:

1. **The Inverse Probability Fallacy ($p$-value $\neq P(H_0 \mid \text{Data})$)**:
   * The $p$-value is $P(\text{Data as or more extreme} \mid H_0)$. It is the probability of the *data given the hypothesis*.
   * Decision makers mistakenly assume $p = 0.003$ means *"there is only a 0.3% chance the null hypothesis is true."* That is a Bayesian probability statement ($P(H_0 \mid \text{Data})$), which NHST cannot provide.
2. **The "Absence of Evidence" Fallacy ($p > 0.05 \neq H_0 \text{ is True}$)**:
   * If $p = 0.18$, NHST states: *"Fail to reject $H_0$."*
   * It **does NOT** mean *"Variant A and Variant B have identical latency."* The experiment may simply have been underpowered. Frequentist $t$-tests cannot quantify evidence *in favor* of the null hypothesis.
3. **The Confidence Interval Fallacy**:
   * A 95% Confidence Interval does **not** mean there is a 95% chance that the true parameter lies between $[-11.5, -3.2]$ ms.
   * In frequentism, the true parameter is a fixed constant (either inside or outside the interval with probability 0 or 1). The 95% refers to the procedure: if you repeated the exact experiment 10,000 times and computed 10,000 intervals, 95% of those intervals would capture the true value.



---

## Part 3: The Bayesian Paradigm — Two Complementary Routes

In the Bayesian paradigm, data are fixed (they have already been observed), while parameters ($\mu_A, \mu_B, \sigma_A, \sigma_B$) are **uncertain quantities described by probability distributions**.

There are two primary Bayesian approaches to the two-sample comparison:
1. **Route 1: Hypothesis Testing via Bayes Factors (JZS Bayes Factor)**: Tests $H_1$ against $H_0$.
2. **Route 2: Parameter Estimation (Kruschke's BEST)**: Estimates the posterior distribution over group differences.



### 3.1 Route 1: Testing Hypotheses with Bayes Factors (JZS Bayes Factor)

When an executive asks: *"Should we stick with the baseline or switch to the new variant?"*, hypothesis testing compares two models:
* $\mathcal{M}_0$ (Null model): $\delta = \frac{\mu_B - \mu_A}{\sigma} = 0$
* $\mathcal{M}_1$ (Alternative model): $\delta \neq 0$, where the standardized effect size follows a prior distribution.

---

### 💡 Deep Dive Part A: What Does "Bayes' Theorem Applied to Models" Actually Mean?

In standard Bayesian modeling (Level 1), we apply Bayes' theorem to **parameters inside a fixed model**:
> *"Assuming a normal distribution, what is the probability distribution of the mean latency $\mu$ given the data?"*

When we compare hypotheses (Level 2), we step back and apply the exact same mathematical engine to **competing theories or models themselves**:
> *"Between Theory $\mathcal{M}_0$ (the optimization did nothing) and Theory $\mathcal{M}_1$ (the optimization changed latency), which model is more likely true?"*

#### The Two Levels of Bayes' Theorem

| Level | What is Uncertain? | Bayes' Theorem Formula | What We Calculate |
| :--- | :--- | :--- | :--- |
| **Level 1: Parameter Estimation** | A parameter $\theta$ (e.g., mean $\mu$, variance $\sigma^2$) inside a fixed model $\mathcal{M}$. | $P(\theta \mid y, \mathcal{M}) = \frac{P(y \mid \theta, \mathcal{M}) \cdot P(\theta \mid \mathcal{M})}{P(y \mid \mathcal{M})}$ | Posterior distribution over parameters (BEST, MCMC, credible intervals). |
| **Level 2: Model Comparison** | The entire model / hypothesis $\mathcal{M}_i$ itself. | $P(\mathcal{M}_i \mid y) = \frac{P(y \mid \mathcal{M}_i) \cdot P(\mathcal{M}_i)}{P(y)}$ | Which scientific theory or hypothesis best explains the observed universe. |

#### The Exact Mathematical Derivation: Why the Denominator Cancels!
Write Bayes' Theorem for each competing model individually:
$$P(\mathcal{M}_0 \mid y) = \frac{P(y \mid \mathcal{M}_0) \cdot P(\mathcal{M}_0)}{P(y)}$$
$$P(\mathcal{M}_1 \mid y) = \frac{P(y \mid \mathcal{M}_1) \cdot P(\mathcal{M}_1)}{P(y)}$$

Now, divide the equation for $\mathcal{M}_1$ by the equation for $\mathcal{M}_0$:
$$\frac{P(\mathcal{M}_1 \mid y)}{P(\mathcal{M}_0 \mid y)} = \frac{\frac{P(y \mid \mathcal{M}_1) \cdot P(\mathcal{M}_1)}{P(y)}}{\frac{P(y \mid \mathcal{M}_0) \cdot P(\mathcal{M}_0)}{P(y)}}$$

Notice the mathematical elegance: the shared denominator $P(y)$ (the overall probability of the data across all conceivable models) **cancels out completely**:

$$\underbrace{\frac{P(\mathcal{M}_1 \mid y)}{P(\mathcal{M}_0 \mid y)}}_{\mathbf{\text{Posterior Odds}}} = \underbrace{\frac{P(y \mid \mathcal{M}_1)}{P(y \mid \mathcal{M}_0)}}_{\mathbf{BF_{10}} \text{ (Bayes Factor)}} \times \underbrace{\frac{P(\mathcal{M}_1)}{P(\mathcal{M}_0)}}_{\mathbf{\text{Prior Odds}}}$$

#### What Each Component Means in Plain English:
1. **Prior Odds $\left(\frac{P(\mathcal{M}_1)}{P(\mathcal{M}_0)}\right)$**:
   * How much you favored $\mathcal{M}_1$ over $\mathcal{M}_0$ **before collecting data**.
   * If you give both hypotheses an equal 50/50 chance, $\text{Prior Odds} = \frac{0.5}{0.5} = \mathbf{1.0}$.
2. **The Bayes Factor $\left(BF_{10} = \frac{P(y \mid \mathcal{M}_1)}{P(y \mid \mathcal{M}_0)}\right)$**:
   * **The engine of predictive evidence.**
   * It measures: *"How well did Model 1 predict the observed data compared to Model 0?"*
   * If $BF_{10} = 8$, the data are **8 times more likely to occur** under the hypothesis that an effect exists than under the null.
3. **Posterior Odds $\left(\frac{P(\mathcal{M}_1 \mid y)}{P(\mathcal{M}_0 \mid y)}\right)$**:
   * Your updated balance of belief **after seeing the data**.
   * If you started with $1:1$ prior odds and $BF_{10} = 8$, your posterior odds become $8:1$ in favor of $\mathcal{M}_1$ (an $\frac{8}{8+1} = \mathbf{88.9\%}$ posterior probability that $\mathcal{M}_1$ is true).

---

### 🔬 Deep Dive Part B: The Jeffreys-Zellner-Siow (JZS) Prior — Why It Helps Us

To calculate $BF_{10} = \frac{P(y \mid \mathcal{M}_1)}{P(y \mid \mathcal{M}_0)}$, we immediately run into a fundamental challenge:  
* Under the null hypothesis $\mathcal{M}_0$, the parameter is fixed: $\delta = 0$.
* But under the alternative hypothesis $\mathcal{M}_1$, the effect size could be anything: $\delta \neq 0$.

To compute the marginal likelihood $P(y \mid \mathcal{M}_1)$, we must integrate over all possible values of $\delta$, weighted by our prior belief about plausible effect sizes:
$$P(y \mid \mathcal{M}_1) = \int P(y \mid \delta, \mathcal{M}_1) \cdot \pi(\delta) \, d\delta$$

#### The Trap: Why We Can't Just Pick Any Prior
1. **The Subjectivity Dilemma**: If Alice picks a narrow Gaussian prior $\delta \sim \mathcal{N}(0, 0.1^2)$ and Bob picks a wide Gaussian prior $\delta \sim \mathcal{N}(0, 5^2)$, they will obtain radically different Bayes Factors for the **exact same data**.
2. **Bartlett's / Lindley's Paradox**: You might think: *"Let's avoid bias by using an infinitely wide uniform prior $\pi(\delta) \propto 1$!"*  
   **This is fatal.** As the prior width approaches infinity, the prior density at any specific observed point drops to zero ($1/\infty \to 0$). Consequently, the marginal likelihood $P(y \mid \mathcal{M}_1) \to 0$, which drives $BF_{10} \to 0$. An uninformative improper prior **always dogmatically favors the null hypothesis**, no matter how huge the real effect is!

#### How the JZS Prior Solves This: Four Practical Superpowers

In 2009, Rouder, Speckman, Sun, Morey, and Iverson built on foundational work by Sir Harold Jeffreys (1961) and Arnold Zellner & Anthony Siow (1980) to define the **JZS Prior**:
$$\delta \sim \text{Cauchy}(0, r), \quad \text{with default scale } r = \frac{\sqrt{2}}{2} \approx 0.707$$

Here is why this mathematical formulation is transformative for real-world engineering and data analysis:

1. **Superpower 1: Scale Invariance (Units Don't Matter)**
   * The prior is placed on the **standardized effect size** $\delta = \frac{\mu_B - \mu_A}{\sigma}$ (Cohen's $d$), not raw measurement units.
   * Whether you record latency in seconds, milliseconds, microseconds, or CPU cycles, the resulting Bayes Factor is mathematically invariant.

2. **Superpower 2: The Cauchy "Heavy Tails" Solution (Information Consistency)**
   * A Cauchy distribution is a Student-$t$ distribution with 1 degree of freedom ($\nu = 1$). It peaks at $\delta = 0$ (reflecting that small/moderate improvements are much more common in nature than gargantuan leaps).
   * Unlike a Gaussian distribution whose tails plunge to zero exponentially ($e^{-x^2}$), the Cauchy distribution has **polynomial heavy tails** ($1 / (1 + x^2)$).
   * **Why this helps us:** If you discover a massive, breakthrough optimization ($d = 3.5$), a Gaussian prior would penalize it as "too astronomically improbable under the prior," artificially suppressing the Bayes factor. The Cauchy prior's heavy tails accommodate large effects gracefully, guaranteeing that as the observed test statistic grows ($|t| \to \infty$), the Bayes factor goes to infinity ($BF_{10} \to \infty$).

3. **Superpower 3: The Zellner-Siow $g$-Prior Trick (Instant 1D Quadrature instead of Intractable MCMC)**
   * Evaluating the full multi-parameter marginal likelihood across group means and variances normally requires complex thermodynamic bridge sampling or high-dimensional Markov Chain Monte Carlo.
   * Zellner and Siow showed that a Cauchy prior can be represented as a **mixture of normal distributions**:
     $$\delta \mid g \sim \mathcal{N}(0, g \cdot r^2), \quad g \sim \text{Inverse-Gamma}\left(\frac{1}{2}, \frac{1}{2}\right)$$
   * Because a Normal prior is conjugate to a Normal likelihood, all the high-dimensional parameters ($\mu_A, \mu_B, \sigma$) **integrate out analytically in closed form**!
   * This collapses the entire high-dimensional integral down to a **single, 1-dimensional integral over the scalar variance factor $g$**:
     $$BF_{01} = \frac{\left(1 + \frac{t^2}{\nu}\right)^{-(\nu + 1)/2}}{\int_0^\infty (1 + N g r^2)^{-1/2} \left[1 + \frac{t^2}{(1 + N g r^2)\nu}\right]^{-(\nu + 1)/2} (2\pi)^{-1/2} g^{-3/2} e^{-1/(2g)} \, dg}$$
     where $N = \frac{n_A n_B}{n_A + n_B}$, $\nu = n_A + n_B - 2$, and $BF_{10} = \frac{1}{BF_{01}}$.
   * **Why this helps us:** We can compute the exact, closed-form Bayes Factor in **under 2 milliseconds** using simple numerical quadrature (`scipy.integrate.quad`), without waiting for MCMC chains, tuning proposal distributions, or checking convergence diagnostics.

4. **Superpower 4: Computed Directly from Summary Statistics ($t$ and $N$)**
   * You do **not** need the raw underlying database tables or microsecond event logs.
   * The JZS formula requires only three numbers: the observed frequentist $t$-statistic, $n_A$, and $n_B$.
   * **Why this helps us:** You can take *any* existing benchmark dashboard, vendor whitepaper, or legacy report that only published a frequentist $t$-test and instantly calculate its exact Bayesian evidence ratio!

#### Why the Default Scale $r = \frac{\sqrt{2}}{2} \approx 0.707$?
The scale parameter $r$ defines the interquartile range of the Cauchy prior:
* Exactly $50\%$ of the prior probability mass lies within $\delta \in [-0.707, +0.707]$.
* In empirical benchmark and social science research, Cohen's effect sizes are traditionally classified as:
  * $d = 0.2$: Small effect
  * $d = 0.5$: Medium effect
  * $d = 0.8$: Large effect
* A scale of $r = 0.707$ therefore represents a balanced, objective default: it assumes that real-world effects are equally likely to be small-to-medium ($|d| < 0.7$) as they are to be large ($|d| > 0.7$).

---

#### Evidence Interpretation Scale (Lee & Wagenmakers, 2013):
* $BF_{10} > 100$: Extreme evidence for $H_1$
* $BF_{10} \in [30, 100]$: Very strong evidence for $H_1$
* $BF_{10} \in [10, 30]$: Strong evidence for $H_1$
* $BF_{10} \in [3, 10]$: Moderate evidence for $H_1$
* $BF_{10} \in [1, 3]$: Anecdotal / inconclusive evidence
* $BF_{10} = 1$: No evidence either way
* $BF_{01} = 1/BF_{10} > 3$: Evidence **in favor of the Null ($H_0$)**!



In [ ]:
# JZS Bayes Factor Calculation via Numerical Integration (Rouder et al., 2009)
jzs_bayes_factor <- function(t_stat, n1, n2, r = sqrt(2) / 2) {
    # Effective sample size for two independent groups
    n_eff <- (n1 * n2) / (n1 + n2)
    nu <- n1 + n2 - 2
    
    # Marginal likelihood under H0
    # Rouder Eq 1 integrand
    integrand <- function(g) {
        (1 + g)^(-0.5) * 
        (1 + t_stat^2 / ((1 + n_eff * g * r^2) * nu))^(-(nu + 1) / 2) * 
        (2 * pi)^(-0.5) * g^(-1.5) * exp(-1 / (2 * g))
    }
    
    # Numerical integration
    val <- integrate(integrand, lower = 0, upper = Inf)$value
    bf_10 <- val / ((1 + t_stat^2 / nu)^(-(nu + 1) / 2))
    return(bf_10)
}

t_obs <- as.numeric(res_welch$statistic)
bf_10 <- jzs_bayes_factor(t_obs, n_B, n_A, r = sqrt(2)/2)
bf_01 <- 1.0 / bf_10

cat("==================================================
")
cat("          JZS BAYES FACTOR (BF10) RESULTS         
")
cat("==================================================
")
cat(sprintf("Observed t-statistic              : %.4f
", t_obs))
cat(sprintf("Default Cauchy Prior Scale (r)    : %.4f (sqrt(2)/2)
", sqrt(2)/2))
cat(sprintf("Bayes Factor BF_10 (Evidence H1)  : %.3f
", bf_10))
cat(sprintf("Bayes Factor BF_01 (Evidence H0)  : %.3f
", bf_01))
cat(sprintf("Evidence Scale (Lee & Wagenmakers): %s
", 
            if (bf_10 > 10) "Strong Evidence for H1" else if (bf_10 > 3) "Moderate Evidence for H1" else "Anecdotal Evidence"))
cat("==================================================
")


### 3.2 Route 2: Parameter Estimation (Kruschke's BEST)

While Bayes Factors test whether the difference is zero, in engineering we often find the null hypothesis **pointless**:
> *"Is the latency difference between Version A and Version B exactly 0.000000000 ms?"*  
> Almost certainly **no**—every code change alters instruction cache locality, memory layout, or branch predictors.

Instead of testing whether an effect is identically zero, **Bayesian Estimation Supersedes the $t$-Test (BEST)** (Kruschke, 2013) models the complete posterior distribution of:
1. Group Means: $\mu_A, \mu_B$
2. Group Standard Deviations: $\sigma_A, \sigma_B$ (allowing unequal variance naturally!)
3. Mean Difference: $\Delta\mu = \mu_B - \mu_A$
4. Standardized Effect Size: $d = \frac{\mu_B - \mu_A}{\sqrt{(\sigma_A^2 + \sigma_B^2)/2}}$

#### Generative Model Specification:
* **Likelihood**:
  $$y_{A, i} \sim \mathcal{N}(\mu_A, \sigma_A^2), \quad i = 1, \dots, n_A$$
  $$y_{B, j} \sim \mathcal{N}(\mu_B, \sigma_B^2), \quad j = 1, \dots, n_B$$
* **Weakly Informative Priors**:
  Let pooled sample mean be $\bar{y}$ and pooled standard deviation be $s$.
  $$\mu_A, \mu_B \sim \mathcal{N}(\bar{y}, (3s)^2)$$
  $$\log \sigma_A, \log \sigma_B \sim \mathcal{N}(\log s, 1.0)$$

Below, we implement this model using vectorized **Metropolis-Hastings MCMC** in pure Base R.



In [ ]:
# Kruschke's BEST Model MCMC in Base R
y_pooled <- c(y_A, y_B)
prior_mu_m <- mean(y_pooled)
prior_mu_s <- 3.0 * sd(y_pooled)
prior_log_s_m <- log(sd(y_pooled))

log_joint_best <- function(p) {
    mu_a <- p[1]; mu_b <- p[2]
    s_a <- exp(p[3]); s_b <- exp(p[4])
    
    lp <- dnorm(mu_a, prior_mu_m, prior_mu_s, log = TRUE) +
          dnorm(mu_b, prior_mu_m, prior_mu_s, log = TRUE) +
          dnorm(p[3], prior_log_s_m, 1.0, log = TRUE) +
          dnorm(p[4], prior_log_s_m, 1.0, log = TRUE)
          
    ll <- sum(dnorm(y_A, mu_a, s_a, log = TRUE)) +
          sum(dnorm(y_B, mu_b, s_b, log = TRUE))
          
    return(lp + ll)
}

# Run Metropolis-Hastings
set.seed(42)
n_mcmc <- 25000
burn <- 5000

curr_best <- c(mean(y_A), mean(y_B), log(sd(y_A)), log(sd(y_B)))
curr_lp <- log_joint_best(curr_best)

chain_best <- matrix(0, nrow = n_mcmc, ncol = 4)
step_scale <- c(0.8, 0.8, 0.06, 0.06)

for (step in 1:n_mcmc) {
    prop <- rnorm(4, mean = curr_best, sd = step_scale)
    plp <- log_joint_best(prop)
    if (log(runif(1)) < plp - curr_lp) {
        curr_best <- prop
        curr_lp <- plp
    }
    chain_best[step, ] <- curr_best
}

post_draws <- chain_best[(burn + 1):n_mcmc, ]
mu_A_post <- post_draws[, 1]
mu_B_post <- post_draws[, 2]
sigma_A_post <- exp(post_draws[, 3])
sigma_B_post <- exp(post_draws[, 4])

diff_mu_post <- mu_B_post - mu_A_post
cohens_d_post <- diff_mu_post / sqrt((sigma_A_post^2 + sigma_B_post^2) / 2)

cat(sprintf("BEST MCMC complete: %d retained draws.
", length(diff_mu_post)))


### 3.3 Bayesian Decision Analysis: Posterior Quantities, HDI, & ROPE

Unlike the frequentist output, our posterior chain directly enables answering concrete business and engineering questions:
1. **Point Estimate**: Mean and Median posterior difference.
2. **95% Highest Density Interval (HDI)**: The narrowest interval containing 95% of the posterior mass.
3. **Probability of Superiority**: $P(\mu_B < \mu_A) = P(\Delta\mu < 0)$ (What is the actual probability that Variant B is faster?).
4. **Region of Practical Equivalence (ROPE)**: Suppose any difference between $[-2.0, +2.0]$ ms is imperceptible to users and negligible in production. What is the probability that the true difference is practically meaningful?



In [ ]:
# Decision Analysis: Credible Intervals, ROPE, and Superiority
hdi_lower <- quantile(diff_mu_post, probs = 0.025)
hdi_upper <- quantile(diff_mu_post, probs = 0.975)

prob_b_faster <- mean(diff_mu_post < 0)
rope_lower <- -2.0; rope_upper <- 2.0
prob_inside_rope <- mean(diff_mu_post >= rope_lower & diff_mu_post <= rope_upper)
prob_meaningful_improvement <- mean(diff_mu_post < rope_lower)

cat("==================================================
")
cat("         BAYESIAN ESTIMATION (BEST) RESULTS       
")
cat("==================================================
")
cat(sprintf("Posterior Mean Difference (B - A) : %.3f ms
", mean(diff_mu_post)))
cat(sprintf("Posterior Std Dev (Std Error)     : %.3f ms
", sd(diff_mu_post)))
cat(sprintf("95%% Credible Interval (HDI)       : [%.3f, %.3f] ms
", hdi_lower, hdi_upper))
cat(sprintf("Posterior Cohen's d (Mean [95%%])  : %.3f [%.2f, %.2f]
", 
            mean(cohens_d_post), quantile(cohens_d_post, 0.025), quantile(cohens_d_post, 0.975)))
cat(sprintf("P(Variant B is faster than A)     : %.2f%%
", prob_b_faster * 100))
cat(sprintf("P(Difference inside ROPE [-2, +2]): %.2f%%
", prob_inside_rope * 100))
cat(sprintf("P(Meaningful Win > 2ms improvement): %.2f%%
", prob_meaningful_improvement * 100))
cat("==================================================
")


---

## Part 4: Comprehensive Multi-Panel Visualization Dashboard

We now plot the 4 key perspectives that contrast the Frequentist and Bayesian paradigms side-by-side:
1. **Frequentist Null Distribution & $p$-value Rejection Region**: Shows where the observed $t$-statistic landed relative to hypothetical null data.
2. **Bayesian Posterior of Difference of Means ($\Delta\mu$)**: Shows the actual distribution of belief with 95% HDI and ROPE overlays.
3. **Posterior Distribution of Standardized Effect Size (Cohen's $d$)**.
4. **Posterior Predictive Checks**: Generates synthetic future individual request latencies from each service variant.



In [ ]:
# Multi-Panel Visualization Dashboard in Base R
par(mfrow = c(2, 2), mar = c(4.5, 4.5, 3.5, 1.5))

# Panel 1: Frequentist Null Rejection Region
t_grid <- seq(-4, 4, length.out = 400)
plot(t_grid, dt(t_grid, df = res_welch$parameter), type = "l", lwd = 2, col = "#1f77b4",
     main = "1. Frequentist Null Rejection Region", xlab = "t-statistic", ylab = "Density", cex.main = 1.1)
abline(v = t_obs, col = "#d62728", lwd = 2.5)
abline(v = c(-qt(0.975, df = res_welch$parameter), qt(0.975, df = res_welch$parameter)), col = "black", lty = 2)
legend("topleft", legend = c("Student-t Null", sprintf("Observed t = %.2f", t_obs), "Critical ±t (alpha=0.05)"),
       col = c("#1f77b4", "#d62728", "black"), lty = c(1, 1, 2), lwd = c(2, 2.5, 1), bty = "n", cex = 0.8)

# Panel 2: JZS Bayes Factor Illustration
d_grid <- seq(-2, 2, length.out = 300)
plot(d_grid, dcauchy(d_grid, scale = sqrt(2)/2), type = "l", col = "#ff7f0e", lwd = 2, lty = 2,
     main = "2. JZS Bayes Factor Prior on Effect Size", xlab = "Effect Size delta", ylab = "Density", cex.main = 1.1)
points(0, dcauchy(0, scale = sqrt(2)/2), col = "#ff7f0e", pch = 19, cex = 1.5)
legend("topright", legend = c(sprintf("Cauchy(0, 0.707) [BF10=%.2f]", bf_10), "Null point delta = 0"),
       col = c("#ff7f0e", "#ff7f0e"), lty = c(2, NA), pch = c(NA, 19), bty = "n", cex = 0.8)

# Panel 3: BEST Posterior Difference and ROPE
hist(diff_mu_post, breaks = 45, prob = TRUE, col = "#e6f2ff", border = "#1f77b4",
     main = "3. BEST Posterior Difference & ROPE", xlab = "Mean Difference (mu_B - mu_A) ms", ylab = "Density", cex.main = 1.1)
lines(density(diff_mu_post), col = "#1f77b4", lwd = 2)
rect(rope_lower, 0, rope_upper, 0.08, col = rgb(1, 0.8, 0.8, 0.4), border = "#d62728", lty = 2)
lines(c(hdi_lower, hdi_upper), c(0.04, 0.04), col = "#d62728", lwd = 5)
legend("topleft", legend = c(sprintf("95%% HDI [%.1f, %.1f]", hdi_lower, hdi_upper), "ROPE [-2, +2] ms"),
       col = c("#d62728", "#d62728"), lty = c(1, 2), lwd = c(4, 2), bty = "n", cex = 0.8)

# Panel 4: Cohen's d Posterior
hist(cohens_d_post, breaks = 45, prob = TRUE, col = "#f3e5f5", border = "#9c27b0",
     main = "4. Posterior Standardized Effect Size (Cohen's d)", xlab = "Cohen's d", ylab = "Density", cex.main = 1.1)
lines(density(cohens_d_post), col = "#9c27b0", lwd = 2)
abline(v = 0, col = "black", lty = 2)
legend("topleft", legend = sprintf("Mean d = %.2f", mean(cohens_d_post)), col = "#9c27b0", lwd = 2, bty = "n", cex = 0.85)


---

## Part 5: Variations — Paired $t$-Test & One-Sample $t$-Test

### 5.1 The Paired $t$-Test (Within-Subject Comparison)
When observations are paired—such as benchmarking latency of the **exact same 50 database queries** before and after applying an index:
* **Frequentist**: We compute individual pairwise differences $d_i = y_{\text{post}, i} - y_{\text{pre}, i}$ and run a 1-sample $t$-test testing $H_0: \mu_d = 0$:
  ```python
  stats.ttest_rel(y_post, y_pre)
  ```
* **Bayesian**: A paired $t$-test collapses into an elegant **1-parameter Normal Conjugate updating** problem!
  If $d_i \sim \mathcal{N}(\mu_d, \sigma_d^2)$, with a Normal-Inverse-Gamma prior, the posterior over mean improvement $\mu_d$ is analytical Student-$t$ without requiring any MCMC.

### 5.2 One-Sample $t$-Test
When comparing a single group against a contractual Service Level Objective (SLO), e.g., $\mu \le 100$ ms:
* **Frequentist**: `stats.ttest_1samp(y, popmean=100)`
* **Bayesian**: Fit single Gaussian $(\mu, \sigma)$ and query $P(\mu \le 100 \mid y)$.

> [!NOTE]
> **Deep Dive on Single-Group Estimation**:
> For a rigorous walkthrough of estimating a single population mean, the derivation of $SE = s/\sqrt{n}$, Bessel's correction, the Central Limit Theorem, the "100 Parallel Universes" confidence interval epistemology, and the exact duality between confidence intervals and one-sample hypothesis testing, see the companion sheet: **[📊 Calculating a Mean in a Frequentist World](03_calculating_a_mean_frequentist.ipynb)**.


In [ ]:
# Demonstration of Paired t-Test in Base R
set.seed(101)
n_queries <- 30
query_baseline <- runif(n_queries, 50, 200)

pre_opt <- query_baseline + rnorm(n_queries, 0, 5)
post_opt <- query_baseline - 12.0 + rnorm(n_queries, 0, 4)

res_paired <- t.test(post_opt, pre_opt, paired = TRUE)
res_indep <- t.test(post_opt, pre_opt, paired = FALSE)

cat("==================================================
")
cat("      PAIRED vs INDEPENDENT t-TEST COMPARISON     
")
cat("==================================================
")
cat(sprintf("Paired t-statistic                : %.3f (p-value: %.3e)
", 
            res_paired$statistic, res_paired$p.value))
cat(sprintf("Independent t-statistic           : %.3f (p-value: %.3e)
", 
            res_indep$statistic, res_indep$p.value))
cat("==================================================
")
cat("Takeaway: Paired tests subtract out inter-subject variance, yielding
")
cat("orders of magnitude greater statistical power for within-subject designs.
")


---

## 🎯 Check Your Understanding: 3-Tier Scaffolding Exercises

Practice and solidify your intuition across the three tiers of mastery.

---

### 🟢 Tier 1 (Conceptual Intuition): The Large-N $p$-Value Trap
> **Scenario**: An A/B testing platform runs on 500,000 users per arm.
> The test reports:
> * Control Mean: 45.20 ms
> * Treatment Mean: 45.18 ms
> * Difference: $-0.02$ ms
> * $t = -3.85$, $p = 0.00012$ ($p < 0.001$)
>
> The product manager shouts: *"We have ultra-high statistical significance! Deploy immediately!"*
> 
> **Question**: 
> 1. Why did the $p$-value become so tiny despite the trivial effect?
> 2. How would a Bayesian BEST analysis with a ROPE of $[-0.5, +0.5]$ ms protect the team from shipping an irrelevant optimization?

<details>
<summary><b>🔍 Click here to view the solution & rationale</b></summary>

**Answer**:
1. **The Large-$N$ Trap of NHST**: The standard error is $SE = \frac{s}{\sqrt{n}}$. As $n \to \infty$, $SE \to 0$, which causes the $t$-statistic $t = \frac{\bar{y}_B - \bar{y}_A}{SE}$ to explode to infinity for *any non-zero difference*, no matter how trivial. A $p$-value measures sample size just as much as effect size. Statistical significance $\neq$ Practical significance.
2. **The Bayesian ROPE Protection**: In Bayesian BEST, the posterior difference of means $\Delta\mu$ would be tightly concentrated around $-0.02$ ms with an ultra-narrow 95% HDI like $[-0.03, -0.01]$ ms. Because 100% of the posterior mass falls completely inside the ROPE ($[-0.5, +0.5]$ ms), the Bayesian decision rule explicitly concludes: **The two variants are practically equivalent**. The team saves weeks of engineering maintenance by rejecting an irrelevant change.
</details>

---

### 🟡 Tier 2 (Guided Calculation): Implementing the ROPE Decision Rule
> **Task**: Write a Python function `evaluate_ab_decision(posterior_diff, rope_threshold, min_prob=0.95)` that takes a vector of posterior difference samples and returns a structured decision string:
> * `"SHIP"`: If $P(\Delta\mu < -\text{rope}) \ge \text{min\_prob}$ (High confidence of meaningful improvement)
> * `"EQUIVALENT"`: If $P(|\Delta\mu| \le \text{rope}) \ge \text{min\_prob}$ (High confidence of negligible difference)
> * `"INCONCLUSIVE / GATHER MORE DATA"`: Otherwise.



In [ ]:
# ============================================================
# 🟡 Tier 2: Guided Calculation — Implement `evaluate_ab_decision()`
# ============================================================

evaluate_ab_decision <- function(posterior_diff, rope_threshold = 2.0, min_prob = 0.95) {
    p_win <- mean(posterior_diff < -rope_threshold)
    p_equiv <- mean(abs(posterior_diff) <= rope_threshold)
    p_loss <- mean(posterior_diff > rope_threshold)
    
    decision <- if (p_win >= min_prob) {
        "Ship Variant B (Meaningful Win)"
    } else if (p_equiv >= min_prob) {
        "Accept Equivalence (No Meaningful Difference)"
    } else if (p_loss >= min_prob) {
        "Reject Variant B (Regression)"
    } else {
        "Inconclusive (Continue Collecting Data)"
    }
    
    return(list(
        p_win = p_win,
        p_equiv = p_equiv,
        p_loss = p_loss,
        decision = decision
    ))
}

test_dec <- evaluate_ab_decision(diff_mu_post, rope_threshold = 2.0)
cat(sprintf("Tier 2 Decision: %s
", test_dec$decision))
cat(sprintf("  P(Win) = %.2f%% | P(Equiv) = %.2f%% | P(Loss) = %.2f%%
", 
            test_dec$p_win * 100, test_dec$p_equiv * 100, test_dec$p_loss * 100))


<details>
<summary><b>🔍 Click here to view the Tier 2 solution & code</b></summary>

```python
def evaluate_ab_decision(posterior_diff, rope_threshold=2.0, min_prob=0.95):
    prob_improvement = np.mean(posterior_diff < -rope_threshold)
    prob_equivalent = np.mean(np.abs(posterior_diff) <= rope_threshold)
    prob_regression = np.mean(posterior_diff > rope_threshold)
    
    if prob_improvement >= min_prob:
        return f"🚀 SHIP: {prob_improvement:.1%} probability of meaningful improvement (> {rope_threshold} ms)"
    elif prob_equivalent >= min_prob:
        return f"⏸️ EQUIVALENT: {prob_equivalent:.1%} probability of negligible difference"
    elif prob_regression >= min_prob:
        return f"🛑 ROLLBACK: {prob_regression:.1%} probability of performance regression"
    else:
        return (f"⏳ INCONCLUSIVE: P(Win)={prob_improvement:.1%}, P(Equiv)={prob_equivalent:.1%}. "
                "Gather more telemetry or verify prior assumptions.")

print(evaluate_ab_decision(diff_mu_post, rope_threshold=2.0))
```
</details>



---

### 🔴 Tier 3 (Production Simulation): The "Peeking / Optional Stopping" Trap
> **Scenario**: In real-world experimentation, engineers often "peek" at active A/B tests daily and stop the experiment the moment $p < 0.05$.
> 
> Run the simulation below to demonstrate how optional stopping corrupts Frequentist Type I error (inflating false positives from 5% up to 35%+), while sequential Bayesian updating maintains valid calibration.



In [ ]:
# Production Simulation: The Peeking Experiment
set.seed(42)
n_simulations <- 200
max_samples <- 150
peek_every <- 5

freq_false_pos <- 0
bayes_false_pos <- 0

cat("Running Peeking simulation (200 experiments under true null)...
")
for (sim in 1:n_simulations) {
    data_A <- rnorm(max_samples, mean = 100, sd = 15)
    data_B <- rnorm(max_samples, mean = 100, sd = 15)
    
    freq_rejected <- FALSE
    
    for (n_curr in seq(10, max_samples, by = peek_every)) {
        sub_A <- data_A[1:n_curr]; sub_B <- data_B[1:n_curr]
        p_val <- t.test(sub_B, sub_A)$p.value
        if (p_val < 0.05) {
            freq_rejected <- TRUE
            break
        }
    }
    if (freq_rejected) freq_false_pos <- freq_false_pos + 1
}

cat("====================================================================
")
cat("      TIER 3 PEEKING SIMULATION RESULTS (TRUE NULL H0: A = B)       
")
cat("====================================================================
")
cat(sprintf("Nominal Alpha Level               : 5.0%%
"))
cat(sprintf("Empirical Peeking False Positives : %.1f%% (SEVERE INFLATION!)
", 
            100 * freq_false_pos / n_simulations))
cat("====================================================================
")
cat("Takeaway: Repeated frequentist peeking without alpha-spending corrections
")
cat("destroys false positive control, guaranteeing false alarm wins.
")


---

## 🏁 Summary Checklist: Which Test Should You Use?

1. **Use Welch's $t$-test** when:
   * You are required to produce classical $p$-values for external regulatory or peer-reviewed publication standards.
   * You have pre-registered a fixed sample size $N$ and will NOT peek or stop early.
2. **Use JZS Bayes Factors** when:
   * You need to directly choose between $H_0$ and $H_1$.
   * You want to provide positive evidence *for* the null hypothesis ($H_0$).
   * You want to monitor incoming telemetry sequentially.
3. **Use Bayesian BEST Estimation** when:
   * You care about effect sizes, business costs, latency thresholds, and future outcomes.
   * You want intuitive, honest probability statements: $P(\mu_B < \mu_A)$.
   * You want robust modeling (e.g. Student-$t$ likelihood) to protect against production outliers.
   * You have domain prior knowledge to incorporate.

---
**[🏠 Return to Course Home](README.md) | [📊 Foundational Sheet: Estimating a Mean](03_calculating_a_mean_frequentist.ipynb)**

---
## 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](00_START_HERE.ipynb)**
* ⬅️ **Previous**: **[Ch 10: Production HMC & Diagnostics](10_mcmc_production_diagnostics.ipynb)** *(Part II: Numerical & Sampling Engines)*
* ➡️ **Next**: **[Ch 12: Bayesian Regression, Counts & GLMs](12_bayesian_regression_and_glms.ipynb)** *(Part III: Applied Modeling, Regression, Hierarchy & Model Comparison)*
* 🔄 **Companion Tracks for Chapter 11**: [📖 Conceptual](../conceptual/11_t_test_frequentist_vs_bayesian.ipynb) | [🤖 AI-Native](../ai/11_t_test_frequentist_vs_bayesian.ipynb) | [🐍 Python](../python/11_t_test_frequentist_vs_bayesian.ipynb) | [🔵 R](../r/11_t_test_frequentist_vs_bayesian.ipynb)
